In [1]:
import dataset
from langchain_core.prompts import PromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_classic.evaluation import ExactMatchStringEvaluator

def pretty_print(input):
    print("QUESTION:")

    q = input['question']
    for line in q.split('\n'):
        print('\t', line)

    if 'documents' in input:
        print("DOCUMENTS")
        for i, doc in enumerate(input['documents'].split('\n\n---\n\n')):
            print(f'{i}:', '\t', doc)
            print()
    

def default_template():
    template = """
    Respond to the question with a single letter: A,B,C or D. Only output the letter of the correct answer.
    Question: {question}
    Answer:
    """.strip()
    return template

def gen_model(model='gemma2:2b', template=None):
    if template is None:
        template = default_template()

    prompt = PromptTemplate.from_template(template)
    # deterministic model
    model = OllamaLLM(model=model,temperature=0,num_predict=1,top_k=10,top_p=0.5, keep_alive="10m")
    chain = prompt | model

    def predict(question, **kwargs):
        print_input = kwargs.pop('print_input', False)
        inputs = {"question": question, **kwargs}
        
        if print_input:
            pretty_print(inputs)

        return chain.invoke(inputs).strip()

    return predict

def gen_question(item):
    options = [f'{k}: {v}' for k,v in item['choices'].items()]
    question = item['question']
    return '\n'.join([question] + options)

def gen_evaluator():
    evaluator = ExactMatchStringEvaluator()
    return lambda ans, ref: evaluator.evaluate_strings(prediction=ans,reference=ref)['score']

def dataloader(path='data/questions.jsonl'):
    for item in (dataset.load_questions(path)):
        yield gen_question(item), item['answer']

def eval_model(
    model, 
    dataset, 
    print_input=False, 
    print_answer=False,
    print_iterative_scores=False,
    ):

    scorer = gen_evaluator()
    ok = 0
    for i, (q, ref) in enumerate(dataset):
        ans = model(q, print_input=print_input)
        sc = scorer(ans, ref)
        if sc == 1.0: 
            ok += 1
        acc = round(100 * ok / (i+1), 2)

        if print_answer:
            print('answered:', ans, 'correct:', ref)
            print('--')
        if print_iterative_scores:
            print('round', i, '| acc:', acc)
            print('--')
    
    return acc

##### rag

def rag_template():
    template = """
    Answer the multiple-choice question using the retrieved documents as the main evidence.

    Retrieved documents:
    {documents}

    Question:
    {question}

    Output only one letter: A, B, C, or D.
    Exactly one character from this set: A B C D.
    Do not write words. Do not explain.

    Answer:
    """.strip()
    return template


import wiki
from collections import defaultdict

def group_by_title(docs):
    ''' input: document chunks '''
    ''' extracts the contents and gruops by document title'''
    groups = defaultdict(list)
    for doc in docs:
        groups[doc.metadata['title']] += [doc.page_content]
    return groups

def build_context(docs):
    groups = group_by_title(docs)
    sections = []
    for title, chunks in groups.items():
        sections += [
            f'# {title}\n\n' + '\n\n'.join(chunks)
        ]
    return "\n\n---\n\n".join(sections)

def rag_input_builder(retriever):
    def build(q):
        docs = retriever(q)
        context = build_context(docs)
        return {
            'question': q,
            'documents': context,
        }
    return build

def gen_rag_model(retriever):
    model = gen_model(template=rag_template())
    input_builder = rag_input_builder(retriever)
    return lambda q, **kwargs: model(**input_builder(q), **kwargs)


In [ ]:
configs = [
    {'k': 5, 'chunk_size': 250, 'chunk_overlap': 50},
    {'k': 2, 'chunk_size': 200, 'chunk_overlap': 50},
    {'k': 3, 'chunk_size': 300, 'chunk_overlap': 100},
    {'k': 1, 'chunk_size': 900, 'chunk_overlap': 100},
]

bm25 = wiki.retriever(**configs[-1])
# model = gen_model()
model = gen_rag_model(bm25)
ds = dataloader('data/questions2.jsonl')
ds = list(ds)
ds = [ds[id-1] for id in [14, 25]]

acc = eval_model(
    model, ds, 
    print_input=True,
    print_answer=True,
    print_iterative_scores=True, 
    )
acc

QUESTION:
	 What plural suffix did Strachevsky’s 1848 dictionary identify as being added in the Talysh dialect?
	 A: -un
	 B: -an
	 C: -ha
	 D: -i
DOCUMENTS
0: 	 # talysh language

The first information about the Talysh language in Russian can be found in Volume X of Strachevsky's "Encyclopedic Dictionary" ("Справочный энциклопедический словарь"), published in St. Petersburg in 1848, which says:

> The Talysh dialect is one of the six main dialects of Persian. It is used in the Talysh khanate and is probably the homeland of that language. Due to its grammatical and lexicographic forms, this language is noticeably different from other dialects.  Except for the addition of the plural suffix "un", it is peculiar and is not derived from any Pahlavi or any other language. This language puts all relative pronouns before the noun, and the pronouns themselves are original in it.

answered: A correct: A
--
round 0 | acc: 100.0
--
QUESTION:
	 What Yiddish term is used for beet sour in the borsch

100.0

In [ ]:
bm25 = wiki.retriever(k=5)
q = 'Tehran'

In [ ]:


docs = bm25(q)
context = build_context(docs)

defaultdict(<class 'list'>, {'tehran': ['| Tehran  تهران | |\n| --- | --- |\n| Capital city | |\n| Tehran skyline and the Alborz  Milad Tower and Mount Damavand  Azadi Tower  Golestan Palace  National Garden  Ferdows Garden  Niavaran Complex  Grand Bazaar  Tabiat Bridge | |\n| Flag of Tehran Flag  Official seal of Tehran Seal | |\n| Map Map of Tehran | |\n| Tehran is located in Iran Tehran  Tehran  Location in Iran and Asia Show map of Iran  Tehran is located in Middle East Tehran  Tehran  Tehran (Middle East) Show map of Middle East  Tehran is located in Asia Tehran  Tehran  Tehran (Asia) Show map of Asia | |\n| Coordinates: 35°41′20″N 51°23′23″E\ufeff / \ufeff35.6889°N 51.3897°E\ufeff / 35.6889; 51.3897 | |\n| Country | Iran |\n| Province | Tehran |', '* ![South Korea](/wiki/South_Korea "South Korea") Seoul, South Korea\n\nSee also\n--------\n\n* Iran International Exhibitions Company\n* Islamic City Council of Tehran\n* List of people from Tehran\n* Tehran City Council (1968–1979) "

In [19]:
context

['# tehran\n\n| Tehran  تهران | |\n| --- | --- |\n| Capital city | |\n| Tehran skyline and the Alborz  Milad Tower and Mount Damavand  Azadi Tower  Golestan Palace  National Garden  Ferdows Garden  Niavaran Complex  Grand Bazaar  Tabiat Bridge | |\n| Flag of Tehran Flag  Official seal of Tehran Seal | |\n| Map Map of Tehran | |\n| Tehran is located in Iran Tehran  Tehran  Location in Iran and Asia Show map of Iran  Tehran is located in Middle East Tehran  Tehran  Tehran (Middle East) Show map of Middle East  Tehran is located in Asia Tehran  Tehran  Tehran (Asia) Show map of Asia | |\n| Coordinates: 35°41′20″N 51°23′23″E\ufeff / \ufeff35.6889°N 51.3897°E\ufeff / 35.6889; 51.3897 | |\n| Country | Iran |\n| Province | Tehran |\n\n* ![South Korea](/wiki/South_Korea "South Korea") Seoul, South Korea\n\nSee also\n--------\n\n* Iran International Exhibitions Company\n* Islamic City Council of Tehran\n* List of people from Tehran\n* Tehran City Council (1968–1979) "Tehran City Council (1968–1